# STaR on Qwen2.5 Instruct at three sizes: linear equations

This notebook fine-tunes `Qwen/Qwen2.5-0.5B-Instruct`, `Qwen/Qwen2.5-1.5B-Instruct` and
`Qwen/Qwen2.5-3B-Instruct` with STaR (Zelikman et al. 2022, Algorithm 1) on DeepMind Mathematics
`algebra__linear_1d` and measures test accuracy after every round. The three models run one after
another on the same splits, prompts and settings. Only the micro-batch changes with size; the
batch stays 32.

| | Condition | Trained on |
|---|---|---|
| A | Base | nothing (3-shot prompt) |
| B | STaR | its own chains that reach the answer key, plus hinted retries that reach it |
| C | STaR without rationalization | its own chains that reach the answer key |
| D | Answer-only | every training problem's answer, no reasoning |

Generation uses vLLM. Fine-tuning uses TRL's `SFTTrainer`, full weights in fp32 with bf16 mixed
precision, so the 3B model needs an 80 GB GPU.

| Mode | What it does | GPU |
|---|---|---|
| `PLAN` | loads the data, builds and checks the splits, prints the prompts | none |
| `SMOKE` | for each model: calibration, then 2 rounds of B on 1,000 train and 1,000 test problems; prints a FULL estimate per model | yes |
| `FULL` | conditions A to D for each model | yes |
| `REPLAY` | every table and figure from saved results | none |

The calibration gate is printed for each model and does not stop the run. Each model writes to
its own folder under `OUT_ROOT`, and `MODELS` sets which models run and in what order.

On Colab, run Step 1 first in a fresh runtime: installing vLLM replaces the preinstalled `torch`.

### Step 1: Install dependencies, imports and configuration

In [ ]:
#@title Installation and Setup
RUN_MODE = "PLAN"   # "PLAN", "SMOKE", "FULL" or "REPLAY"

import sys
if "google.colab" in sys.modules:
    !pip install -q datasets
    if RUN_MODE in ("SMOKE", "FULL"):
        !pip install -q vllm trl

import gc
import hashlib
import json
import os
import random
import re
import shutil
import time
from pathlib import Path

os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")   # PyTorch sampler; needs no CUDA compiler

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset

MODELS = {                     # model: micro-batch, in the order they run
    "Qwen/Qwen2.5-0.5B-Instruct": 16,
    "Qwen/Qwen2.5-1.5B-Instruct": 8,
    "Qwen/Qwen2.5-3B-Instruct": 4,
}
MODEL_ID = next(iter(MODELS))
DATASET = "deepmind/math_dataset"
MODULE = "algebra__linear_1d"
SEED = 0

# Splits
N_DEV = 500
N_TRAIN = 10_000
N_TEST = 10_000                # the whole official test split

# STaR
ROUNDS = 6
EPOCHS = 2
LR = 1e-5
BATCH_SIZE = 32
MICRO_BATCH = 16
WARMUP_RATIO = 0.05
MAX_NEW_TOKENS = 256
VLLM_MEMORY = 0.5              # share of GPU memory vLLM may take

# Calibration gate (dev split, base model)
CALIB_SAMPLES = 16
CALIB_TEMPERATURE = 0.8
GATE_GREEDY = (0.20, 0.60)
GATE_MIN_GAP = 0.15
GATE_MIN_PARSE = 0.90
OVERRIDE_GATE = True           # print the gate for each model, keep going

if RUN_MODE == "SMOKE":
    N_TRAIN, N_TEST, ROUNDS, EPOCHS = 1_000, 1_000, 2, 1

USE_DRIVE = "google.colab" in sys.modules and RUN_MODE in ("SMOKE", "FULL", "REPLAY")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = Path("/content/drive/MyDrive/sapiotomy/page_12_ft_instruct_sizes")
else:
    OUT_ROOT = Path("out/page_12_ft_instruct_sizes")
if RUN_MODE == "SMOKE":
    OUT_ROOT = OUT_ROOT / "smoke"
CKPT_ROOT = Path("/content/ckpt") if "google.colab" in sys.modules else OUT_ROOT / "ckpt"


def use_model(model_id):
    """Point every later step at one model: its weights, micro-batch, results folder and checkpoints."""
    global MODEL_ID, MICRO_BATCH, OUT_DIR, CKPT_DIR
    MODEL_ID, MICRO_BATCH = model_id, MODELS[model_id]
    slug = model_id.split("/")[-1]
    OUT_DIR, CKPT_DIR = OUT_ROOT / slug, CKPT_ROOT / slug
    for d in (OUT_DIR, OUT_DIR / "traces", CKPT_DIR):
        d.mkdir(parents=True, exist_ok=True)


use_model(MODEL_ID)

assert RUN_MODE in ("PLAN", "SMOKE", "FULL", "REPLAY"), RUN_MODE
if RUN_MODE in ("SMOKE", "FULL"):
    assert torch.cuda.is_available(), "SMOKE and FULL need a GPU runtime"
    print(torch.cuda.get_device_name(0))
random.seed(SEED)
print(f"RUN_MODE={RUN_MODE}  OUT_ROOT={OUT_ROOT}")
print("models", ", ".join(f"{m} (micro-batch {b})" for m, b in MODELS.items()))

### Step 2: Load the data and build the splits

The Hugging Face copy is read from its converted parquet files. Questions and answers arrive
wrapped as `b'...\n'` and are cleaned. Dev and train are drawn from the official train split;
every evaluation uses the official test split.

In [ ]:
def unwrap(s):
    s = s.strip()
    if s[:2] in ("b'", 'b"'):
        s = s[2:-1]
    return s.replace("\\n", "").strip()


raw = load_dataset(DATASET, revision="refs/convert/parquet",
                   data_files={"train": f"{MODULE}/train/0000.parquet",
                               "test": f"{MODULE}/test/0000.parquet"})
print({k: len(v) for k, v in raw.items()})

EXEMPLARS = [
    ("Solve 24 = 1601*c - 1605*c for c.",
     "Combine the c terms: 1601*c - 1605*c = -4*c, so 24 = -4*c. Divide both sides by -4: c = -6. "
     "The answer is -6."),
    ("Solve 657 = -220*t + 1086*t + 22307 for t.",
     "Combine the t terms: -220*t + 1086*t = 866*t, so 657 = 866*t + 22307. Subtract 22307 from "
     "both sides: -21650 = 866*t. Divide both sides by 866: t = -25. The answer is -25."),
    ("Solve -11*y - 263*y + 3162 = -88*y for y.",
     "Combine the y terms on the left: -274*y + 3162 = -88*y. Add 274*y to both sides: "
     "3162 = 186*y. Divide both sides by 186: y = 17. The answer is 17."),
]

test_all = [{"q": unwrap(q), "a": unwrap(a)} for q, a in zip(raw["test"]["question"], raw["test"]["answer"])]
test_qs = {r["q"] for r in test_all}
exemplar_qs = {q for q, _ in EXEMPLARS}

train_q, train_a = raw["train"]["question"], raw["train"]["answer"]
order = list(range(len(train_q)))
random.Random(SEED).shuffle(order)
picked, seen = [], set()
for i in order:
    q = unwrap(train_q[i])
    if q in seen or q in exemplar_qs or q in test_qs:
        continue
    seen.add(q)
    picked.append({"q": q, "a": unwrap(train_a[i]), "idx": i})
    if len(picked) == N_DEV + N_TRAIN:
        break

SPLITS = {
    "dev": picked[:N_DEV],
    "train": picked[N_DEV:],
    "test": random.Random(SEED).sample(test_all, N_TEST) if N_TEST < len(test_all) else test_all,
}

qsets = {k: {r["q"] for r in v} for k, v in SPLITS.items()}
assert not qsets["dev"] & qsets["train"]
assert not (qsets["dev"] | qsets["train"]) & test_qs
assert not exemplar_qs & (qsets["dev"] | qsets["train"] | qsets["test"])
HASHES = {k: hashlib.sha256("\n".join(r["q"] for r in v).encode()).hexdigest()[:16] for k, v in SPLITS.items()}
for k, v in SPLITS.items():
    print(f"{k:6s} {len(v):6d}  hash {HASHES[k]}  e.g. {v[0]['q']}  ->  {v[0]['a']}")

### Step 3: Prompts and answers

Every chain ends in `The answer is N.` The hinted prompt adds `(the answer is N)` after the
question; the hint is never stored with the chain.

In [ ]:
FEWSHOT = "".join(f"Q: {q}\nA: {a}\n\n" for q, a in EXEMPLARS)
ANSWER = re.compile(r"The answer is\s*(-?\d+)\.?")
HINT_MENTION = re.compile(r"\(the answer is|we are told|given that the answer|hint", re.I)


def question(q):
    return f"Q: {q}\nA:"


def prompt(q, fewshot=True, hint=None):
    h = f" (the answer is {hint})" if hint is not None else ""
    return (FEWSHOT if fewshot else "") + f"Q: {q}{h}\nA:"


def chain(text):
    """The text up to and including the answer sentence, and the answer; (None, None) if absent."""
    m = ANSWER.search(text)
    if not m:
        return None, None
    return text[: m.end()].strip().rstrip(".") + ".", m.group(1)


print(prompt(SPLITS["train"][0]["q"], hint=SPLITS["train"][0]["a"]))

### Step 4: Generation with vLLM

`Model` loads weights from the Hub or a checkpoint folder the first time it generates.
`close()` frees the GPU before fine-tuning.

In [ ]:
TIMINGS = {"load": [], "gen": [], "train": []}


class Model:
    """vLLM on one set of weights, loaded on first use so cached stages never load it."""

    def __init__(self, weights):
        self.weights, self.llm = weights, None

    def get(self):
        if self.llm is None:
            from vllm import LLM
            t0 = time.time()
            self.llm = LLM(str(self.weights), dtype="bfloat16", gpu_memory_utilization=VLLM_MEMORY,
                           max_model_len=2048, seed=SEED, enable_prefix_caching=True)
            TIMINGS["load"].append(time.time() - t0)
        return self.llm

    def close(self):
        if self.llm is not None:
            self.llm.llm_engine.engine_core.shutdown()
            self.llm = None
            gc.collect()
            torch.cuda.empty_cache()


def generate(model, prompts, n=1, temperature=0.0):
    from vllm import SamplingParams
    params = SamplingParams(n=n, temperature=temperature, max_tokens=MAX_NEW_TOKENS,
                            stop=["\nQ:"], seed=SEED)
    t0 = time.time()
    outs = model.get().generate(prompts, params)
    TIMINGS["gen"].append((len(prompts) * n, time.time() - t0))
    return [[c.text.strip() for c in o.outputs] for o in outs]


def grade(rows, texts, **tags):
    """One record per generated text: the chain, its answer, and whether it matches the key."""
    recs = []
    for r, outs in zip(rows, texts):
        for t in outs:
            c, a = chain(t)
            recs.append({**tags, "q": r["q"], "key": r["a"], "text": t, "chain": c,
                         "answer": a, "correct": a == r["a"]})
    return recs

### Step 5: Fine-tuning with SFTTrainer

Each call starts from the base weights and trains on `(prompt, completion)` pairs, with the loss
on the completion only. Weights are kept in fp32 with bf16 mixed precision.

In [ ]:
def finetune(examples, out):
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer
    t0 = time.time()
    ds = Dataset.from_list([{"prompt": p, "completion": c} for p, c in examples])
    args = SFTConfig(
        output_dir=str(out) + "_work",
        learning_rate=LR,
        lr_scheduler_type="linear",
        warmup_steps=WARMUP_RATIO,         # a float below 1 is a share of all steps
        per_device_train_batch_size=MICRO_BATCH,
        gradient_accumulation_steps=BATCH_SIZE // MICRO_BATCH,
        num_train_epochs=EPOCHS,
        bf16=True,
        gradient_checkpointing=True,
        completion_only_loss=True,
        model_init_kwargs={"dtype": torch.float32},
        max_length=512,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        seed=SEED,
    )
    trainer = SFTTrainer(model=MODEL_ID, args=args, train_dataset=ds)
    trainer.train()
    trainer.save_model(str(out))
    losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    shutil.rmtree(str(out) + "_work", ignore_errors=True)
    TIMINGS["train"].append((len(examples) * EPOCHS, time.time() - t0))
    return {"examples": len(examples), "first_loss": losses[0] if losses else None,
            "last_loss": losses[-1] if losses else None}

### Step 6: Saving results

Every stage writes a JSONL file to `OUT_DIR/traces`. A stage that already has its file is
loaded instead of run, so a dropped session resumes where it stopped.

In [ ]:
def path(tag):
    return OUT_DIR / "traces" / f"{tag}.jsonl"


def load(tag):
    return [json.loads(l) for l in path(tag).open()]


def stage(tag, fn):
    if path(tag).exists():
        return load(tag)
    recs = fn()
    tmp = path(tag).with_suffix(".tmp")
    with tmp.open("w") as f:
        for r in recs:
            f.write(json.dumps(r) + "\n")
    tmp.rename(path(tag))
    return recs


def accuracy(recs):
    return sum(r["correct"] for r in recs) / len(recs)


def parse_rate(recs):
    return sum(r["answer"] is not None for r in recs) / len(recs)

### Step 7: Calibration gate

The base model on the dev split, 3-shot: greedy accuracy, and whether any of 16 samples reaches
the key (pass@16).

In [ ]:
def calibration(model):
    dev = SPLITS["dev"]
    g = stage("calib_greedy", lambda: grade(dev, generate(model, [prompt(r["q"]) for r in dev]),
                                            kind="greedy"))
    s = stage("calib_sampled", lambda: grade(
        dev, generate(model, [prompt(r["q"]) for r in dev], n=CALIB_SAMPLES, temperature=CALIB_TEMPERATURE),
        kind="sampled"))
    hit = {}
    for r in s:
        hit[r["q"]] = hit.get(r["q"], False) or r["correct"]
    row = {"pass@1": accuracy(g), f"pass@{CALIB_SAMPLES}": sum(hit.values()) / len(hit),
           "parsed": parse_rate(g)}
    row["gap"] = row[f"pass@{CALIB_SAMPLES}"] - row["pass@1"]
    ok = (GATE_GREEDY[0] <= row["pass@1"] <= GATE_GREEDY[1] and row["gap"] >= GATE_MIN_GAP
          and row["parsed"] >= GATE_MIN_PARSE)
    print("  ".join(f"{k} {v:.1%}" for k, v in row.items()), "->", "PASS" if ok else "FAIL")
    row["gate_passed"] = ok
    assert ok or OVERRIDE_GATE, "calibration gate failed; see the plan, section 8"
    return row

### Step 8: STaR

Round `r` loads model `r` once, evaluates it on test, and generates the chains that train model
`r + 1`. Model 0 is the base model with the 3-shot prompt; fine-tuned models are evaluated
without it. Each fine-tune restarts from the base weights. A run stops early if test accuracy
falls two rounds in a row.

In [ ]:
def evaluate(model, fewshot, tag):
    test = SPLITS["test"]
    return stage(tag, lambda: grade(test, generate(model, [prompt(r["q"], fewshot) for r in test]),
                                    kind="eval"))


def collect(model, fewshot, rationalize, tag):
    """Greedy chains on train; for B, a hinted retry of every miss. Returns training records."""
    train = SPLITS["train"]
    greedy = stage(f"{tag}_greedy", lambda: grade(
        train, generate(model, [prompt(r["q"], fewshot) for r in train]), kind="greedy"))
    kept = [r for r in greedy if r["correct"]]
    if rationalize:
        missed = [{"q": r["q"], "a": r["key"]} for r in greedy if not r["correct"]]
        hinted = stage(f"{tag}_hinted", lambda: grade(
            missed, generate(model, [prompt(r["q"], fewshot, hint=r["a"]) for r in missed]), kind="hinted"))
        kept += [r for r in hinted if r["correct"]]
    return kept


def ckpt(cond, r):
    return CKPT_DIR / f"{cond}_r{r}"


def run_star(cond):
    log = []
    for r in range(ROUNDS + 1):
        model = Model(MODEL_ID if r == 0 else ckpt(cond, r))
        tag = "A_r0" if r == 0 else f"{cond}_r{r}"
        ev = evaluate(model, r == 0, f"{tag}_eval")
        row = {"condition": cond, "round": r, "test_acc": accuracy(ev), "parsed": parse_rate(ev)}
        log.append(row)
        print(f"{cond} round {r}: test accuracy {row['test_acc']:.1%}")
        acc = [x["test_acc"] for x in log]
        if r == ROUNDS or (r >= 2 and acc[-1] < acc[-2] < acc[-3]):
            model.close()
            break
        kept = collect(model, r == 0, cond == "B", f"{tag}_train")
        model.close()
        row["next_train_greedy"] = sum(k["kind"] == "greedy" for k in kept)
        row["next_train_hinted"] = sum(k["kind"] == "hinted" for k in kept)
        row["hinted_mentioning_hint"] = sum(bool(HINT_MENTION.search(k["chain"]))
                                            for k in kept if k["kind"] == "hinted")
        if not ckpt(cond, r + 1).exists():
            stats = finetune([(question(k["q"]), " " + k["chain"]) for k in kept], ckpt(cond, r + 1))
            print(f"  trained model {r + 1} on {stats['examples']} chains, "
                  f"loss {stats['first_loss']:.3f} -> {stats['last_loss']:.3f}")
        if r >= 1:
            shutil.rmtree(ckpt(cond, r), ignore_errors=True)   # keep only the latest model
    return log


def run_answer_only():
    if not path("D_r1_eval").exists() and not ckpt("D", 1).exists():
        finetune([(question(r["q"]), f" The answer is {r['a']}.") for r in SPLITS["train"]], ckpt("D", 1))
    model = Model(ckpt("D", 1))
    ev = evaluate(model, False, "D_r1_eval")
    model.close()
    print(f"D: test accuracy {accuracy(ev):.1%}")
    return [{"condition": "D", "round": 1, "test_acc": accuracy(ev), "parsed": parse_rate(ev)}]

### Step 9: Speed and manifest

`write_manifest` records one model's speeds, versions and settings next to its results. SMOKE
turns the measured speeds into an estimate for FULL.

In [ ]:
def write_manifest(res):
    import transformers, trl, vllm, datasets
    from huggingface_hub import HfApi
    gen_rate = sum(n for n, _ in TIMINGS["gen"]) / sum(s for _, s in TIMINGS["gen"])
    train_rate = sum(n for n, _ in TIMINGS["train"]) / sum(s for _, s in TIMINGS["train"])
    load_s = sum(TIMINGS["load"]) / len(TIMINGS["load"])
    print(f"generation {gen_rate:.0f} completions/s, training {train_rate:.0f} examples/s, "
          f"vLLM load {load_s:.0f} s")
    if RUN_MODE == "SMOKE":
        n_train, n_test, rounds, epochs, keep = 10_000, 10_000, 6, 2, 0.6
        per_round = ((n_train * 1.75 + n_test) / gen_rate          # greedy, hinted retries, eval
                     + n_train * keep * epochs / train_rate + load_s)
        answer_only = n_train * epochs / train_rate + n_test / gen_rate + load_s
        calib = N_DEV * (1 + CALIB_SAMPLES) / gen_rate + load_s
        est = calib + 2 * rounds * per_round + answer_only
        print(f"FULL estimate for {MODEL_ID} on this GPU: {est / 3600:.1f} h "
              f"({per_round / 60:.1f} min per STaR round)")
    manifest = {
        "model": MODEL_ID, "model_revision": HfApi().model_info(MODEL_ID).sha,
        "dataset": f"{DATASET}:{MODULE}", "split_hashes": HASHES,
        "versions": {m.__name__: m.__version__ for m in (torch, transformers, trl, vllm, datasets)},
        "gpu": torch.cuda.get_device_name(0), "run_mode": RUN_MODE,
        "config": {k: globals()[k] for k in ("MICRO_BATCH", "SEED", "N_DEV", "N_TRAIN", "N_TEST", "ROUNDS", "EPOCHS",
                                            "LR", "BATCH_SIZE", "WARMUP_RATIO",
                                            "MAX_NEW_TOKENS", "CALIB_SAMPLES", "CALIB_TEMPERATURE")},
        "speed": {"completions_per_s": gen_rate, "train_examples_per_s": train_rate, "vllm_load_s": load_s},
        "seconds": res["seconds"],
    }
    json.dump(manifest, open(OUT_DIR / "manifest.json", "w"), indent=1)

### Step 10: Run

The models run in the order `MODELS` lists them. A stage already on disk is loaded instead of
run, so a dropped session resumes where it stopped.

In [ ]:
RESULTS = {}

if RUN_MODE == "PLAN":
    print(f"Splits are built: {N_DEV} dev, {N_TRAIN} train, {N_TEST} test. "
          f"FULL runs conditions A to D for up to {ROUNDS} rounds on {len(MODELS)} models.")

if RUN_MODE in ("SMOKE", "FULL"):
    for model_id in MODELS:
        use_model(model_id)
        for k in TIMINGS:
            TIMINGS[k] = []
        print(f"\n######## {model_id} (micro-batch {MICRO_BATCH}) ########")
        t_start = time.time()
        res = {}
        model = Model(MODEL_ID)
        res["calibration"] = calibration(model)
        model.close()
        res["B"] = run_star("B")
        if RUN_MODE == "FULL":
            res["C"] = run_star("C")
            res["D"] = run_answer_only()
        res["seconds"] = round(time.time() - t_start)
        json.dump(res, open(OUT_DIR / "results.json", "w"), indent=1)
        write_manifest(res)
        RESULTS[model_id] = res
    json.dump(RESULTS, open(OUT_ROOT / "results_all.json", "w"), indent=1)

if RUN_MODE == "REPLAY":
    for model_id in MODELS:
        f = OUT_ROOT / model_id.split("/")[-1] / "results.json"
        if f.exists():
            RESULTS[model_id] = json.load(open(f))
    print("results for", list(RESULTS))

### Step 11: Results

In [ ]:
def show(model_id, res):
    rows = res["B"] + res.get("C", [])[1:] + res.get("D", [])
    print(f"{'condition':>9s} {'round':>5s} {'test acc':>9s} {'parsed':>7s} {'next: greedy':>13s} {'hinted':>7s}")
    for r in rows:
        cond = "A" if r["round"] == 0 else r["condition"]
        print(f"{cond:>9s} {r['round']:>5d} {r['test_acc']:>9.1%} {r['parsed']:>7.1%} "
              f"{r.get('next_train_greedy', ''):>13} {r.get('next_train_hinted', ''):>7}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    ax = axes[0]
    for cond, label in (("B", "STaR"), ("C", "STaR, no rationalization")):
        if res.get(cond):
            ax.plot([r["round"] for r in res[cond]], [r["test_acc"] for r in res[cond]],
                    marker="o", label=label)
    if res.get("D"):
        ax.axhline(res["D"][0]["test_acc"], ls="--", c="gray", label="answer-only")
    ax.set_xlabel("round (0 = base, 3-shot)")
    ax.set_ylabel("test accuracy")
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.legend()

    ax = axes[1]
    b = [r for r in res["B"] if "next_train_greedy" in r]
    xs = [r["round"] + 1 for r in b]
    ax.bar(xs, [r["next_train_greedy"] for r in b], label="greedy, reached the key")
    ax.bar(xs, [r["next_train_hinted"] for r in b], bottom=[r["next_train_greedy"] for r in b],
           label="hinted retry, reached the key")
    ax.set_xlabel("model trained (round)")
    ax.set_ylabel("chains kept")
    ax.set_title("STaR training set per round")
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.legend()
    fig.suptitle(model_id)
    plt.tight_layout()
    plt.show()

for model_id, res in RESULTS.items():
    if res.get("B"):
        print(f"\n{model_id}")
        show(model_id, res)

if sum(bool(r.get("B")) for r in RESULTS.values()) > 1:
    fig, ax = plt.subplots(figsize=(7, 4))
    for model_id, res in RESULTS.items():
        if not res.get("B"):
            continue
        line, = ax.plot([r["round"] for r in res["B"]], [r["test_acc"] for r in res["B"]], marker="o",
                        label=model_id.split("/")[-1])
        if res.get("D"):
            ax.axhline(res["D"][0]["test_acc"], ls="--", c=line.get_color(), alpha=0.6)
    ax.set_xlabel("round (0 = base, 3-shot)")
    ax.set_ylabel("test accuracy")
    ax.set_title("STaR by model size (dashed: answer-only)")
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.legend()
    plt.tight_layout()
    plt.show()

### Step 12: The same test problems, round by round

The greedy STaR answer to a few test problems at every round, for each model.

In [ ]:
for model_id, res in RESULTS.items():
    if not res.get("B"):
        continue
    use_model(model_id)
    print(f"######## {model_id}\n")
    picks = [r["q"] for r in SPLITS["test"][:3]]
    for q in picks:
        print(q)
        for r in res["B"]:
            tag = "A_r0_eval" if r["round"] == 0 else f"B_r{r['round']}_eval"
            rec = next(x for x in load(tag) if x["q"] == q)
            print(f"  round {r['round']} {'✓' if rec['correct'] else '✗'}  {rec['text'][:220]}")
        print()